Sama Al Adib

Alba García 

Jordi Cantó

# PACMAN

### 1. Introducción y objetivo de la sesión práctica
El objetivo principal de esta sesión práctica es construir un agente inteligente y avanzado para Pac-Man, culminando el aprendizaje de las sesiones anteriores. En la Práctica 2 exploramos algoritmos de búsqueda clásica (BFS, DFS, UCS, A*) para encontrar caminos óptimos en laberintos estáticos. Posteriormente, en la Práctica 3 introdujimos un NeuralAgent puramente reactivo o "greedy" que tomaba decisiones mirando solo un paso hacia el futuro mediante Inteligencia Artificial.

Aunque la red neuronal le daba cierta "intuición", su falta de planificación a largo plazo frente a adversarios en movimiento lo hacía vulnerable a encerramientos y trampas. Para solucionar esto y cumplir los requisitos de la Práctica 3.1, hemos abordado el problema combinando lo mejor de ambos mundos:

1. Mejorar el conocimiento explícito del agente añadiendo nuevas heurísticas.
2. Optimizar el entrenamiento de nuestra red neuronal.
3. Desarrollar un agente híbrido (AlphaBetaNeuralAgent) que combina la planificación a futuro frente a adversarios del algoritmo Minimax con Poda Alfa-Beta y la intuición predictiva de la red neuronal entrenada.

### 2. 
#### Part 1. New Heuristics
El agente base que se nos proporcionó solo tenía en cuenta dos factores: la distancia a la comida más cercana y la proximidad de los fantasmas. Hemos ampliado enormemente la función de evaluación (traditional_evaluation) añadiendo las siguientes heurísticas.

1. Atracción hacia las cápsulas de poder

Implementación: Recompensamos a Pac-Man basándonos en la cercanía a la cápsula más próxima (+ 10.0 / (distancia + 1)) y le penalizamos fuertemente por cada cápsula que deja en el mapa (- 20.0 * len(capsules)).
Justificación y peso: Las cápsulas son el recurso que permite a Pac-Man pasar de presa a cazador. Le damos un peso muy alto (penalización de 20 por cápsula restante) para forzarle a que las priorice, ya que comer un fantasma asustado reporta muchísimos puntos.

2. Evitar el acorralamiento por múltiples fantasmas

Implementación: Contamos cuántos fantasmas activos (no asustados) están a una distancia igual o inferior a 3 casillas. Si hay más de uno, aplicamos una penalización masiva (score -= 600).
Justificación y peso: Estar cerca de un fantasma es peligroso, pero estar cerca de dos a la vez casi siempre significa la muerte por pinzamiento. El peso de -600 es el más alto de todo nuestro sistema de reglas, diseñado para que el agente evite este escenario a toda costa, incluso si eso significa alejarse de la comida.

3. Caza agresiva y gestión del tiempo de los fantasmas asustados

Implementación: Si un fantasma está asustado y lo tenemos justo al lado (distancia 1), damos un bonus enorme (score += 500). Sin embargo, si al fantasma le quedan 2 turnos o menos de estar asustado (scaredTimer <= 2) y estamos más lejos que esos turnos, aplicamos una penalización (score -= 100).
Justificación y peso: Comer un fantasma da 200 puntos fijos, por lo que el bonus de +500 asegura que Pac-Man jamás deje escapar una presa garantizada. La penalización de -100 evita el clásico error de perseguir a un fantasma que se va a volver agresivo justo antes de alcanzarlo.

4. Opciones de huida (Movilidad)

Implementación: Si el agente tiene 3 o más acciones legales posibles en su posición actual, le damos un ligero bonus (score += 15).
Justificación y peso: Queremos que Pac-Man prefiera estar en cruces o espacios abiertos en lugar de meterse en pasillos sin salida. El peso es bajo (+15) porque es un factor de desempate; no queremos que se quede dando vueltas en un cruce ignorando la comida.

5. Penalización por volumen total de comida

Implementación: Restamos puntos por cada bolita de comida que quede en el mapa (score -= 4.0 * len(food)).
Justificación y peso: El objetivo final del juego es limpiar el mapa. Esta heurística proporciona un incentivo constante para terminar la partida rápido. El peso (-4.0) es suficiente para desempatar entre dos caminos iguales, pero no tan alto como para que Pac-Man se suicide yendo a por comida si hay fantasmas cerca.

#### Part 2. Network Training 
Dataset utilizado Generamos un dataset de más de 300 partidas ganadoras jugando en el mapa mediumClassic. Para asegurar la calidad de los datos, solo guardamos las partidas donde Pac-Man logró limpiar el tablero.

Arquitectura y mejoras (Opcional implementado) Nuestra red neuronal (PacmanNet) es un modelo lineal (Fully Connected) adaptado al tamaño de nuestro mapa (220 casillas de entrada para mediumClassic). Para mejorar el rendimiento y evitar el sobreajuste (overfitting), modificamos la arquitectura base introduciendo capas de Dropout con una probabilidad del 0.3.

En cuanto a los hiperparámetros:

Redujimos el Learning Rate a 0.005 (frente al 0.001 original) usando el optimizador Adam.
Entrenamos durante 150 epochs con un tamaño de lote (batch size) de 64.
Además, implementamos una función de balanceo del dataset (Oversampling). Nos dimos cuenta de que en las partidas normales, la acción de quedarse quieto (STOP) o ciertos giros eran muy raros comparados con avanzar en línea recta. Nuestro código clona los ejemplos de las acciones minoritarias hasta igualarlas, garantizando que la red aprenda a ejecutar maniobras evasivas poco comunes cuando sea estrictamente necesario.

#### Part 3. AlphaBetaNeuralAgent
¿Cómo funciona la integración? El AlphaBetaNeuralAgent es nuestro agente híbrido final. En lugar de decidir basándose solo en el tablero actual, simula mentalmente un árbol de posibles movimientos (Pac-Man mueve, los fantasmas responden, Pac-Man vuelve a mover...) hasta una profundidad determinada (depth=2). Usa la Poda Alfa-Beta para descartar rápidamente ramas del árbol que los fantasmas nunca le permitirían alcanzar, ahorrando mucha capacidad de cómputo.

Cuando llega al final de esa simulación mental (los nodos hoja), llama a nuestra función evaluation_combined. Es aquí donde se produce la integración: final_score = (w_heuristic * traditional_score) + (w_neural * neural_score)

Nuestra estrategia: Pesos Dinámicos (Opcional implementado) Decidimos no usar pesos estáticos fijos, sino implementar un sistema de Pesos Dinámicos Contextuales que se adapta a lo que ocurre en el tablero:

Modo Caza: Si detectamos que hay algún fantasma asustado en el tablero, nuestras heurísticas tradicionales son perfectas porque tienen reglas estrictas y matemáticas para cazar y gestionar el tiempo del fantasma. En este estado, asignamos w_heuristic = 0.9 y w_neural = 0.1.
Modo Supervivencia: Si todos los fantasmas son peligrosos, la supervivencia a largo plazo es mucho más difusa para reglas matemáticas simples. Aquí es donde brilla el conocimiento implícito que nuestra red neuronal aprendió de las 220 partidas. En este estado, invertimos los pesos y confiamos en la IA: w_heuristic = 0.3 y w_neural = 0.7.
Gracias a este sistema dinámico y a un código optimizado que carga el modelo .pth una sola vez en memoria mediante un neural_helper, el agente logra un equilibrio excelente entre agresividad programada y supervivencia predictiva.